# Phase 1 — Lesson 19: Complex Numbers for AI
> Faithful study notes built directly from the lesson source. Concepts, code, diagrams, and exercises.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 19: Complex Numbers for AI
**Type:** Learn | **Languages:** Python | **Time:** ~60 minutes
**Prerequisites:** Phase 1, Lessons 01-04



# What You'll Learn

---

## Learning Objectives

- Perform complex arithmetic (add, multiply, divide, conjugate) in both rectangular and polar form
- Apply Euler's formula to convert between complex exponentials and trigonometric functions
- Implement the Discrete Fourier Transform using complex roots of unity
- Explain how complex rotations underlie RoPE and sinusoidal positional encodings in transformers


# Complex Numbers: Rectangular vs Polar

---

A complex number z = a + bi where i² = −1.

**Rectangular form** (a + bi): good for addition/subtraction.
**Polar form** r·e^(iθ): good for multiplication/division — multiply magnitudes, add angles.

Euler's formula: e^(iθ) = cos θ + i·sin θ


In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport mathz1 = 3 + 4jz2 = 1 - 1jprint(f"z1 + z2 = {z1 + z2}")print(f"z1 * z2 = {z1 * z2}")print(f"z1 / z2 = {z1 / z2}")print(f"conj(z1) = {z1.conjugate()}")print(f"|z1| = {abs(z1)}")# Plot on complex planefig, ax = plt.subplots(figsize=(6, 6))for z, c, lab in [(z1, '#e74c3c', '3+4i'), (z2, '#27ae60', '1-1j')]:    ax.quiver(0, 0, z.real, z.imag, angles='xy', scale_units='xy', scale=1, color=c, width=0.005)    ax.text(z.real + 0.1, z.imag + 0.1, lab, color=c, fontweight='bold')ax.set_xlim(-2, 5); ax.set_ylim(-2, 5)ax.axhline(0, color='gray', lw=0.8); ax.axvline(0, color='gray', lw=0.8)ax.set_aspect('equal'); ax.set_title('Complex Numbers on the Complex Plane', fontsize=12, fontweight='bold')ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
import cmathimport math# Euler's formula: e^(i*pi) = -1print(f"e^(i*pi) = {cmath.exp(1j * math.pi)}")print(f"e^(i*pi/2) = {cmath.exp(1j * math.pi / 2)}")# Multiplying by i rotates by 90 degreesz = 3 + 4jzi = z * 1jprint(f"z = {z}")print(f"z * i = {zi}")

# Euler's Formula and Phasors

---

e^(i·θ) = cos(θ) + i·sin(θ) — a complex exponential traces the unit circle.

A phasor e^(i·ω·t) is a rotating arrow. Its real part is cos(ω·t), its imaginary part is sin(ω·t).


In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport matht = np.linspace(0, 4 * np.pi, 200)z = np.exp(1j * 2 * t)fig, ax = plt.subplots(figsize=(6, 6))ax.plot(z.real, z.imag, 'b-', alpha=0.6, label='e^(2it)')ax.plot(np.cos(2*t), np.sin(2*t), 'r--', alpha=0.6, label='Real part (cos)')ax.plot([0, z.real[-1]], [0, z.imag[-1]], 'g-', lw=2, label='Final phasor')ax.add_patch(plt.Circle((0, 0), 1, fill=False, color='gray', linestyle=':'))ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)ax.set_aspect('equal'); ax.grid(True, alpha=0.3)ax.set_title('Complex Exponential as a Rotating Phasor', fontsize=12, fontweight='bold')plt.tight_layout(); plt.show()

# Roots of Unity and the DFT

---

The N-th roots of unity are N points equally spaced on the unit circle:

```
w_k = e^(2·π·i·k/N)  for k = 0, ..., N−1
```

The DFT computes X[k] = Σ x[n]·e^(−2·π·i·k·n/N) — the correlation with each root of unity.


In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport cmathimport mathdef dft(x):    N = len(x)    return np.array([sum(x[n] * cmath.exp(-2j * math.pi * k * n / N) for n in range(N)) for k in range(N)])# DFT of a signal with frequencies 2 and 5N = 64n = np.arange(N)signal = np.sin(2 * np.pi * 2 * n / N) + 0.5 * np.sin(2 * np.pi * 5 * n / N)X = dft(signal)magnitudes = np.abs(X)peaks = np.argsort(magnitudes)[::-1][:5]print(f"Top 5 frequency bins: {peaks}")print(f"Magnitude at bin 2: {magnitudes[2]:.2f}")print(f"Magnitude at bin 5: {magnitudes[5]:.2f}")plt.stem(n[:N//2], magnitudes[:N//2], linefmt='#2c3e50', markerfmt=' ', basefmt=' ')plt.title('DFT Magnitude Spectrum', fontsize=12, fontweight='bold')plt.xlabel('Frequency bin'); plt.ylabel('|X[k]|')plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Roots of Unity

---

Properties of N-th roots of unity:
- Every root has magnitude 1
- The sum of all N roots is 0

These properties make the DFT invertible.


In [ ]:
import numpy as npimport cmathimport mathdef roots_of_unity(N):    return [cmath.exp(2j * math.pi * k / N) for k in range(N)]N = 8roots = roots_of_unity(N)print(f"{N}-th roots of unity:")for k, r in enumerate(roots):    print(f"  w_{k} = {r:.4f}")print(f"\nSum of all roots: {sum(roots):.6f} (should be 0)")print(f"Magnitudes: {[round(abs(r), 6) for r in roots]}")

# Connection to Transformers

---

**Sinusoidal positional encodings** (original Transformer):

```
PE(pos, 2i)   = sin(pos / 10000^(2i/d))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d))
```

Each dimension pair oscillates at a different frequency — a complex exponential at that frequency.

**RoPE** multiplies query/key vectors by complex rotation matrices — relative position becomes a rotation angle.


In [ ]:
import numpy as npimport matplotlib.pyplot as plt# Sinusoidal positional encodingsd_model = 128max_pos = 64pos = np.arange(max_pos)[:, None]i = np.arange(d_model)[None, :]angle_rates = 1 / np.power(10000, (2 * (i // 2)) / d_model)angle_rads = pos * angle_ratesangle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])plt.figure(figsize=(8, 6))plt.imshow(angle_rads[:32, :32], cmap='viridis', aspect='auto')plt.colorbar(label='Encoding value')plt.title('Sinusoidal Positional Encodings (first 32 positions, 32 dims)', fontsize=12, fontweight='bold')plt.xlabel('Dimension'); plt.ylabel('Position')plt.tight_layout(); plt.show()# Verify: dot product depends only on relative positiondef pos_enc(pos, d_model=128):    i = np.arange(d_model)[None, :]    angle_rates = 1 / np.power(10000, (2 * (i // 2)) / d_model)    angle_rads = pos * angle_rates    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])    return angle_rads[0]pe_0 = pos_enc(0)pe_1 = pos_enc(1)pe_5 = pos_enc(5)pe_6 = pos_enc(6)print(f"dot(PE(0), PE(1)) = {np.dot(pe_0, pe_1):.4f}")print(f"dot(PE(5), PE(6)) = {np.dot(pe_5, pe_6):.4f}")print("(Should be equal — dot product depends only on relative position)")

# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** What does multiplying by i do?
> **Explanation:** Rotates a complex number by 90° on the complex plane.

**Q2:** What are the N-th roots of unity?
> **Explanation:** N equally spaced points on the unit circle — e^(2πi·k/N).

**Q3:** How does RoPE use complex numbers?
> **Explanation:** Multiplies query/key vectors by complex rotation matrices — relative position becomes rotation angle.



# Practice Exercises

---

Do these in order:

**Exercise 1:** Verify Euler's formula: e^(iπ) ≈ −1, e^(iπ/2) ≈ i.

**Exercise 2:** Compute the DFT of a pure sine wave; verify peaks at ±frequency bins.



# Key Terms

---

| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Complex number | a + bi where i² = −1 | Real part a, imaginary part b |
| Complex conjugate | a − bi | Mirror across real axis; z·conj(z) = |z|² |
| Modulus | |z| = √(a² + b²) | Distance from origin |
| Argument | arg(z) = atan2(b, a) | Angle from positive real axis |
| Euler's formula | e^(iθ) = cos θ + i·sin θ | Connects exponentials to trigonometry |
| Phasor | e^(i·ω·t) | Rotating complex number representing a sinusoid |
| Roots of unity | e^(2πi·k/N) | N equally spaced points on unit circle |
| DFT | Discrete Fourier Transform | Correlates signal with roots of unity |


# Further Reading

---

- [3Blue1Brown: Euler's formula](https://www.youtube.com/watch?v=mvmuCPvRoWQ)
- [RoFormer: RoPE](https://arxiv.org/abs/2104.09864)
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)


# Next Steps

---

When you're ready, continue to Lesson 20: The Fourier Transform.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
